# 🎓 Module 4: Real Research - Cross-Matching with Gaia

Welcome to the Capstone! You've learned to visualize, scale, and accelerate. Now, let's do science.

**The Goal**: We will take a Hubble image, detect the stars using your GPU/CPU, and then query the **Gaia Space Telescope's Database** to see if our detections match the world's most accurate star catalog.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia
from photutils.detection import DAOStarFinder
from astropy.stats import mad_std

print("🔭 Astronomical Research Environment Initialized.")

## 1. Load Data with WCS
WCS (World Coordinate System) is the magic that tells us which pixel $(x, y)$ corresponds to which spot in the universe (RA, Dec).

In [ ]:
from astropy.utils.data import download_file
image_url = 'https://archive.stsci.edu/pub/vosamp/horsehead.fits'
fits_file = download_file(image_url, cache=True)

with fits.open(fits_file) as hdul:
    wcs = WCS(hdul[0].header)
    data = hdul[0].data.astype(float)

print("✅ Loaded image and calibrated World Coordinate System (WCS).")

## 2. Detect Stars Locally
Using the logic from Module 1 & 3, we'll find stars in our image.

In [ ]:
bkg_sigma = mad_std(data)
daofind = DAOStarFinder(fwhm=3.0, threshold=5.*bkg_sigma)
sources = daofind(data - np.median(data))

# Convert pixel coordinates to Sky Coordinates (RA/Dec)
coords_pix = np.transpose((sources['xcentroid'], sources['ycentroid']))
sky_coords = wcs.pixel_to_world(sources['xcentroid'], sources['ycentroid'])

print(f"✨ Detected {len(sources)} stars locally.")

## 3. Query the Gaia Database
Now we ask the ESA Gaia archive for all stars within the area of our image.

In [ ]:
center_coord = wcs.pixel_to_world(data.shape[1]/2, data.shape[0]/2)
print(f"📡 Querying Gaia Archive around {center_coord}...")

job = Gaia.cone_search_async(center_coord, radius=0.1 * u.deg)
r = job.get_results()

print(f"🛰️ Gaia found {len(r)} known stars in this region!")

## 4. The Final Match
We use `match_to_catalog_sky` to find out how many of our stars are officially recognized.

In [ ]:
gaia_coords = SkyCoord(ra=r['ra'], dec=r['dec'])
idx, d2d, d3d = sky_coords.match_to_catalog_sky(gaia_coords)

# Only consider matches within 1 arcsecond (very close)
matches = d2d < 1 * u.arcsec
print(f"🏁 Success! {np.sum(matches)} stars matched perfectly with the Gaia Catalog.")

plt.figure(figsize=(10, 6))
plt.hist(d2d.arcsec, bins=50, color='cyan', range=(0, 5))
plt.title('Coordinate Accuracy (Arcseconds offset from Gaia)')
plt.xlabel('Offset (arcsec)')
plt.ylabel('Number of Stars')
plt.show()